# 第 8 章 · Transformer Block 与模型组装

> 📄 配套动画网页：`../08_block_model.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第二部分 · 模型

把 RMSNorm、注意力、SwiGLU 拼成一个 Block，堆 8 层，加上词嵌入和输出头——一个完整的 MiniMind 就成型了。

---


## 🎯 本章目标

- 理解 Pre-Norm 残差块的结构
- 掌握 `nn.ModuleList`、`nn.Embedding`、`register_buffer`
- 理解权重共享（weight tying）
- 能从零组装完整模型并算交叉熵损失

---


## 📖 背景：一个 Block 的骨架

```python
x = x + Attn( RMSNorm(x) )      # 注意力子层 + 残差
x = x + FFN ( RMSNorm(x) )      # 前馈子层 + 残差
```

**Pre-Norm**：先归一化再进子层。比 Post-Norm 更容易训练深层网络。
**残差**：`x + f(x)`，让梯度有一条「高速公路」直达浅层。

完整模型：`Embedding → 8 × Block → RMSNorm → lm_head → logits`。

---



## 🖼 官方配图

![把前面各章的零件拼起来，就是这张图——Embedding、8 个 Block、RMSNorm、输出头](../images/LLM-structure.jpg)

*把前面各章的零件拼起来，就是这张图——Embedding、8 个 Block、RMSNorm、输出头*

---


# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 8.1 `nn.Embedding`：查表

In [ ]:
import torch
from torch import nn

emb = nn.Embedding(100, 8)          # 词表 100，每词 8 维
ids = torch.tensor([[1, 5, 9], [2, 3, 4]])
print("输入形状:", ids.shape, " 输出形状:", emb(ids).shape)
print("等价于直接索引:", torch.allclose(emb(ids), emb.weight[ids]))

### 8.2 `nn.ModuleList`：动态数量的层

堆 N 层用 `ModuleList`，它会被正确注册为子模块（普通 list 不会）。

In [ ]:
class Stack(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 4) for _ in range(n)])

s = Stack(3)
print("子模块数量:", len(s.layers))
print("参数会被 optimizer 看到:", len(list(s.parameters())) == 6)

### 8.3 `register_buffer`：不参与训练的常量

RoPE 的 cos/sin 表是常量，用 buffer 注册（跟着 `.to(device)` 走，但不更新）。

In [ ]:
class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("freqs", torch.ones(4, 8), persistent=False)

m = M()
print("在 state_dict 里吗:", "freqs" in m.state_dict())   # persistent=False -> 不在
print("在 parameters 里吗:", len(list(m.parameters())))    # 0
print("buffer:", m.freqs.shape)

### 8.4 权重共享（weight tying）

输入嵌入和输出投影共用同一个矩阵。MiniMind 默认开启。

In [ ]:
class Tied(nn.Module):
    def __init__(self, vocab, d):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.lm_head = nn.Linear(d, vocab, bias=False)
        self.lm_head.weight = self.embed.weight      # 共享！

t = Tied(100, 8)
print("共享后是同一个张量:", t.lm_head.weight is t.embed.weight)
print("参数量（应只有 100*8）:", sum(p.numel() for p in t.parameters()))

### 8.5 交叉熵与 `ignore_index`

`labels` 里为 `-100` 的位置会被跳过——这正是 SFT 里「只算回答部分」的实现方式。

In [ ]:
import torch.nn.functional as F

logits = torch.randn(1, 3, 10)                  # [B, L, vocab]
labels = torch.tensor([[3, -100, 7]])           # 中间位置不计损失

loss = F.cross_entropy(logits.view(-1, 10), labels.view(-1), ignore_index=-100)
print("loss =", loss.item())

# 只有 2 个位置参与计算
print("有效标签数:", (labels != -100).sum().item())

### 8.6 前置依赖：`RMSNorm` / `Attention` / `FeedForward`

组装需要前面几章的零件。下面给一份**精简的参考实现**（注意力这里用标准多头、签名简化成 `forward(x)`），
目的是让本章可以独立测试「组装」这件事本身。如果你前面几章已经写好了，欢迎换成你自己的版本。

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return (self.weight * (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps))).type_as(x)

class Attention(nn.Module):
    """精简版：标准多头注意力，签名 forward(x)"""
    def __init__(self, config):
        super().__init__()
        self.n_heads, self.head_dim = config.num_attention_heads, config.head_dim
        d = self.n_heads * self.head_dim
        self.q_proj = nn.Linear(config.hidden_size, d, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, d, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, d, bias=False)
        self.o_proj = nn.Linear(d, config.hidden_size, bias=False)
    def forward(self, x):
        B, S, _ = x.shape
        q, k, v = (p(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
                   for p in (self.q_proj, self.k_proj, self.v_proj))
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, S, -1))

class FeedForward(nn.Module):
    def __init__(self, config, intermediate_size=None):
        super().__init__()
        d = intermediate_size or config.intermediate_size
        self.gate_proj = nn.Linear(config.hidden_size, d, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, d, bias=False)
        self.down_proj = nn.Linear(d, config.hidden_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

现在把所有零件组装起来。这是本章最重要的一段代码。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 8.7 实现 `MiniMindBlock`

**要求**：
- `input_layernorm` / `post_attention_layernorm`：两个 RMSNorm
- `self_attn`：注意力（用上面的参考实现，签名是 `forward(x)`）
- `mlp`：`FeedForward` 或 `MOEFeedForward`（看 `config.use_moe`）
- `forward(self, x)`：

```python
residual = x
x = self.self_attn(self.input_layernorm(x))
x = x + residual
x = x + self.mlp(self.post_attention_layernorm(x))
return x
```

In [ ]:
# ======== 你的实现：MiniMindBlock ========
#
# 【要求】
#   - 签名：MiniMindBlock(layer_id, config)，forward(self, x) 返回张量
#   - input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
#   - post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
#   - self_attn = Attention(config)
#   - mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)
#   - forward 里注意残差相加的时机
#
# 【可用到的函数 / 方法】
#   - 复用上面的 RMSNorm / Attention / FeedForward
#   - `nn.Module`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：MiniMindBlock ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)

    class Cfg:
        hidden_size = 16
        intermediate_size = 32
        num_attention_heads = 2
        head_dim = 8
        use_moe = False
        rms_norm_eps = 1e-6

    blk = MiniMindBlock(0, Cfg())
    x = torch.randn(2, 5, 16)
    y = blk(x)
    assert tuple(y.shape) == (2, 5, 16), f"输出形状应为 (2, 5, 16)，得到 {tuple(y.shape)}"
    with torch.no_grad():
        blk.self_attn.o_proj.weight.zero_()
        blk.mlp.down_proj.weight.zero_()
    close(blk(x), x, tol=1e-6,
          msg="把两个子层的输出投影置零后，残差应让输出严格等于输入 —— 说明残差连接不对")

check("MiniMindBlock 形状 / 残差连接", _t)

### 8.8 实现 `MiniMindForCausalLM`（组装 + 损失）

**要求**：写出这个类。

```python
class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        # embed_tokens: nn.Embedding(vocab_size, hidden_size)
        # layers: nn.ModuleList([MiniMindBlock(i, config) for i in range(num_hidden_layers)])
        # norm: RMSNorm(hidden_size, eps=config.rms_norm_eps)
        # lm_head: nn.Linear(hidden_size, vocab_size, bias=False)
        # 若 config.tie_word_embeddings：lm_head.weight = embed_tokens.weight

    def forward(self, input_ids, labels=None):
        # 返回 (logits, loss)；没传 labels 时 loss 为 None
```

**为什么要错位**：位置 t 的输出要预测位置 t+1 的词。

```python
x, y = logits[:, :-1, :], labels[:, 1:]
loss = F.cross_entropy(x.reshape(-1, vocab_size), y.reshape(-1), ignore_index=-100)
```

In [ ]:
# ======== 你的实现：MiniMindForCausalLM ========
#
# 【要求】
#   - __init__ 里搭好 embed_tokens / layers / norm / lm_head
#   - tie_word_embeddings 为 True 时，让 lm_head.weight 指向 embed_tokens.weight
#   - forward(self, input_ids, labels=None) 返回 (logits, loss)
#   - hidden = embed_tokens(input_ids)，逐个过 Block，最后过 norm
#   - logits = lm_head(hidden)，形状 [B, S, vocab_size]
#   - 有 labels 时错位一位算交叉熵（ignore_index=-100），否则 loss = None
#
# 【可用到的函数 / 方法】
#   - `nn.Embedding`、`nn.ModuleList`、`nn.Linear`
#   - `F.cross_entropy(..., ignore_index=-100)`
#   - `Tensor.reshape(-1, vocab_size)`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：MiniMindForCausalLM ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)

    class Cfg:
        vocab_size = 64
        hidden_size = 16
        num_hidden_layers = 2
        intermediate_size = 32
        num_attention_heads = 2
        head_dim = 8
        use_moe = False
        rms_norm_eps = 1e-6
        tie_word_embeddings = True

    model = MiniMindForCausalLM(Cfg())
    ids = torch.randint(0, 64, (2, 6))
    out = model(ids)
    logits = out[0] if isinstance(out, tuple) else out
    assert tuple(logits.shape) == (2, 6, 64), f"logits 形状应为 (2, 6, 64)，得到 {tuple(logits.shape)}"
    assert model.lm_head.weight is model.embed_tokens.weight, \
        "tie_word_embeddings=True 时，lm_head.weight 应就是 embed_tokens.weight 本身"
    out2 = model(ids, labels=ids.clone())
    loss = out2[1] if isinstance(out2, tuple) else getattr(out2, "loss", None)
    assert loss is not None, "传了 labels 却没有返回 loss"
    assert loss.dim() == 0 and torch.isfinite(loss), f"loss 应是有限的标量，得到 {loss}"
    loss.backward()
    assert model.embed_tokens.weight.grad is not None, "loss.backward() 之后嵌入层应该有梯度"

check("MiniMindForCausalLM 前向 / 权重共享 / 损失可反传", _t)

### 8.9 验证参数量 ≈ 64M

组装完成后，数一下参数量，应该接近 64M（6400 词表 + 8 层 + 768 维）。

In [ ]:
# ======== 你的实现：verify_model ========
#
# 【要求】
#   - 实例化完整的模型
#   - 统计总参数量（M）
#   - 构造 input_ids = torch.randint(0, 6400, (1, 16))，跑一次前向
#   - 确认 logits 形状是 [1, 16, 6400]
#   - 给它 labels 跑一次，确认能算出 loss
#
# 【可用到的函数 / 方法】
#   - `sum(p.numel() for p in model.parameters()) / 1e6`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能画出完整的模型数据流图
- [ ] Pre-Norm 与 Post-Norm 的区别说清楚了吗？
- [ ] 权重共享省了多少参数？
- [ ] loss 的错位对齐做对了吗？（`logits[:-1]` vs `labels[1:]`）
- [ ] 参数量接近 64M 吗？

**思考题**：如果去掉所有残差连接，训练会发生什么？试试把残差去掉跑几步，观察 loss。
